Steps used:

 imports → stage → load → profile → clean → schema → dimensions → facts → accumulating snapshot → periodic snapshot → reconcile

In [ ]:
import os
import math
import shutil
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import (create_engine, MetaData, Table, Column, Integer,
                        SmallInteger, String, Numeric, Float, Date, DateTime,
                        Text, ForeignKey, PrimaryKeyConstraint, text)

RAW_DIR = "."            # source CSVs live in the notebook's folder
STAGING = "staging"      # raw layer (unmodified copies)
DATABASE_URL = os.getenv("DATABASE_URL", "postgresql+psycopg2://postgres:<pssword>@localhost:5432/olist_dw")

FILES = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

### Stage and copy unmodified

In [3]:
os.makedirs(STAGING, exist_ok=True)
for fname in FILES.values():
    shutil.copyfile(os.path.join(RAW_DIR, fname), os.path.join(STAGING, fname))
print(f"Staged {len(FILES)} files unmodified into '{STAGING}/'.")

Staged 9 files unmodified into 'staging/'.


### Load staged data as dataframes

In [4]:
orders      = pd.read_csv(f"{STAGING}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{STAGING}/olist_order_items_dataset.csv")
payments    = pd.read_csv(f"{STAGING}/olist_order_payments_dataset.csv")
reviews     = pd.read_csv(f"{STAGING}/olist_order_reviews_dataset.csv")
customers   = pd.read_csv(f"{STAGING}/olist_customers_dataset.csv",
                          dtype={"customer_zip_code_prefix": str})
products    = pd.read_csv(f"{STAGING}/olist_products_dataset.csv")
sellers     = pd.read_csv(f"{STAGING}/olist_sellers_dataset.csv",
                          dtype={"seller_zip_code_prefix": str})
geolocation = pd.read_csv(f"{STAGING}/olist_geolocation_dataset.csv",
                          dtype={"geolocation_zip_code_prefix": str})
translation = pd.read_csv(f"{STAGING}/product_category_name_translation.csv")
print("Loaded staged data into DataFrames.")

Loaded staged data into DataFrames.


### Profiling (row counts, data types, nul lvalues and duplicate id's)

In [66]:
EXPECTED_ROWS = {"orders": 99441, "order_items": 112650, "payments": 103886,
                 "reviews": 99224, "customers": 99441, "products": 32951,
                 "sellers": 3095, "geolocation": 1000163, "translation": 71}

print("A. ROW COUNTS (actual vs table_info.md)")
for key, df in [("orders", orders), ("order_items", order_items), ("payments", payments),
                ("reviews", reviews), ("customers", customers), ("products", products),
                ("sellers", sellers), ("geolocation", geolocation), ("translation", translation)]:
    ok = "OK" if len(df) == EXPECTED_ROWS[key] else "!! MISMATCH"
    print(f"   {key:12s} actual={len(df):>9,}  expected={EXPECTED_ROWS[key]:>9,}  {ok}")

print("\nB. DTYPES")
for key, df in [("orders", orders), ("order_items", order_items), ("payments", payments),
                ("reviews", reviews), ("customers", customers), ("products", products),
                ("sellers", sellers), ("geolocation", geolocation), ("translation", translation)]:
    print(f"   [{key}] " + " | ".join(f"{c}:{df[c].dtype}" for c in df.columns))

print("\nC. NULLS (only columns with nulls)")
for key, df in [("orders", orders), ("reviews", reviews), ("products", products)]:
    n = df.isna().sum(); n = n[n > 0]
    print(f"   [{key}]")
    print("    " + (n.to_string().replace("\n", "\n    ") if len(n) else "(none)"))

print("\nD. DUPLICATE review_id (key data-quality finding)")
print(f"   duplicate review_id rows = {int(reviews.duplicated(subset=['review_id']).sum()):,}")

A. ROW COUNTS (actual vs table_info.md)
   orders       actual=   99,441  expected=   99,441  OK
   order_items  actual=  112,650  expected=  112,650  OK
   payments     actual=  103,886  expected=  103,886  OK
   reviews      actual=   99,224  expected=   99,224  OK
   customers    actual=   99,441  expected=   99,441  OK
   products     actual=   32,951  expected=   32,951  OK
   sellers      actual=    3,095  expected=    3,095  OK
   geolocation  actual=1,000,163  expected=1,000,163  OK
   translation  actual=       71  expected=       71  OK

B. DTYPES
   [orders] order_id:str | customer_id:str | order_status:str | order_purchase_timestamp:str | order_approved_at:str | order_delivered_carrier_date:str | order_delivered_customer_date:str | order_estimated_delivery_date:str
   [order_items] order_id:str | order_item_id:int64 | product_id:str | seller_id:str | shipping_limit_date:str | price:float64 | freight_value:float64
   [payments] order_id:str | payment_sequential:int64 | payme

### Profiling: Consistency and Referential Integrity

In [67]:
def orphans(child, ccol, parent, pcol):
    pset = set(parent[pcol].dropna())
    return int((~child[ccol].dropna().isin(pset)).sum())

print("E. CONSISTENCY (value rules)")
checks = [
    ("orders.order_status valid", orders["order_status"].isin(
        ["created", "approved", "invoiced", "processing", "shipped",
         "delivered", "unavailable", "canceled"]).all()),
    ("payments.payment_type valid", payments["payment_type"].isin(
        ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]).all()),
    ("reviews.review_score in 1..5", reviews["review_score"].between(1, 5).all()),
    ("order_items.price >= 0", pd.to_numeric(order_items["price"]).ge(0).all()),
    ("order_items.freight_value >= 0", pd.to_numeric(order_items["freight_value"]).ge(0).all()),
    ("payments.payment_value >= 0", pd.to_numeric(payments["payment_value"]).ge(0).all()),
    ("payments.payment_installments >= 0", pd.to_numeric(payments["payment_installments"]).ge(0).all()),
]
for label, ok in checks:
    print(f"   {'PASS' if ok else 'FAIL'}  {label}")

print("\nF. REFERENTIAL INTEGRITY (orphan counts)")
rel = [
    (orders, "customer_id", customers, "customer_id", "orders.customer_id -> customers"),
    (order_items, "order_id", orders, "order_id", "order_items.order_id -> orders"),
    (order_items, "product_id", products, "product_id", "order_items.product_id -> products"),
    (order_items, "seller_id", sellers, "seller_id", "order_items.seller_id -> sellers"),
    (payments, "order_id", orders, "order_id", "payments.order_id -> orders"),
    (reviews, "order_id", orders, "order_id", "reviews.order_id -> orders"),
]
for child, ccol, parent, pcol, lbl in rel:
    print(f"   orphans={orphans(child, ccol, parent, pcol):>6,}  {lbl}")

E. CONSISTENCY (value rules)
   PASS  orders.order_status valid
   PASS  payments.payment_type valid
   PASS  reviews.review_score in 1..5
   PASS  order_items.price >= 0
   PASS  order_items.freight_value >= 0
   PASS  payments.payment_value >= 0
   PASS  payments.payment_installments >= 0

F. REFERENTIAL INTEGRITY (orphan counts)
   orphans=     0  orders.customer_id -> customers
   orphans=     0  order_items.order_id -> orders
   orphans=     0  order_items.product_id -> products
   orphans=     0  order_items.seller_id -> sellers
   orphans=     0  payments.order_id -> orders
   orphans=     0  reviews.order_id -> orders


### Clean: Leading zeroes for ZIP code

In [68]:
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")
print("Timestamps parsed (null milestones remain NaT).")

Timestamps parsed (null milestones remain NaT).


### Clean: Leading zeroes for ZIP code

In [69]:
for df, col in [(customers, "customer_zip_code_prefix"),
                (sellers, "seller_zip_code_prefix"),
                (geolocation, "geolocation_zip_code_prefix")]:
    df[col] = df[col].astype(str).str.strip().str.zfill(5)
print("ZIP prefixes zero-padded to 5 chars.")

ZIP prefixes zero-padded to 5 chars.


### Clean: Rename Typos and Translation errors

In [70]:
products = products.rename(columns={
    "product_name_lenght":        "product_name_length",
    "product_description_lenght": "product_description_length",
})
translation["product_category_name_english"] = translation["product_category_name_english"].replace({
    "costruction_tools_garden": "construction_tools_garden",
    "costruction_tools_tools":  "construction_tools_tools",
    "home_confort":             "home_comfort",
    "fashio_female_clothing":   "fashion_female_clothing",
    "arts_and_craftmanship":    "arts_and_crafts",
})
print("Renamed typo columns; fixed translation typos.")

Renamed typo columns; fixed translation typos.


### Clean: Review test and product gaps

In [71]:
for c in ["review_comment_title", "review_comment_message"]:
    reviews[c] = reviews[c].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

products["product_category_name"] = products["product_category_name"].fillna("unknown")
for c in ["product_name_length", "product_description_length", "product_photos_qty"]:
    products[c] = products[c].fillna(0).astype(int)
for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[c] = products[c].fillna(products[c].median()).round().astype(int)
if "unknown" not in set(translation["product_category_name"]):
    translation = pd.concat([translation, pd.DataFrame(
        [{"product_category_name": "unknown",
          "product_category_name_english": "unknown"}])], ignore_index=True)
print("Review text normalized; product gaps filled.")

Review text normalized; product gaps filled.


### Clean: drop orphan rows

In [72]:
def drop_orphans(child, ccol, parent, pcol, label):
    pset = set(parent[pcol].dropna())
    mask = child[ccol].isin(pset)
    n = int((~mask).sum())
    print(f"   {'drop' if n else 'ok  '} {n:>6,} orphans  {label}")
    return child[mask].reset_index(drop=True)

orders      = drop_orphans(orders, "customer_id", customers, "customer_id", "orders.customer_id -> customers")
order_items = drop_orphans(order_items, "order_id", orders, "order_id", "order_items.order_id -> orders")
order_items = drop_orphans(order_items, "product_id", products, "product_id", "order_items.product_id -> products")
order_items = drop_orphans(order_items, "seller_id", sellers, "seller_id", "order_items.seller_id -> sellers")
payments    = drop_orphans(payments, "order_id", orders, "order_id", "payments.order_id -> orders")
reviews     = drop_orphans(reviews, "order_id", orders, "order_id", "reviews.order_id -> orders")

   ok        0 orphans  orders.customer_id -> customers
   ok        0 orphans  order_items.order_id -> orders
   ok        0 orphans  order_items.product_id -> products
   ok        0 orphans  order_items.seller_id -> sellers
   ok        0 orphans  payments.order_id -> orders
   ok        0 orphans  reviews.order_id -> orders


### Clean: Drop Duplicates

In [73]:
n_geo = len(geolocation)
geolocation = geolocation.drop_duplicates(
    subset=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    keep="first").reset_index(drop=True)
print(f"geolocation: {n_geo:,} -> {len(geolocation):,} ({n_geo - len(geolocation):,} dup coords)")

orders      = orders.drop_duplicates().reset_index(drop=True)
order_items = order_items.drop_duplicates().reset_index(drop=True)
payments    = payments.drop_duplicates().reset_index(drop=True)
reviews     = reviews.drop_duplicates().reset_index(drop=True)
customers   = customers.drop_duplicates().reset_index(drop=True)
products    = products.drop_duplicates().reset_index(drop=True)
sellers     = sellers.drop_duplicates().reset_index(drop=True)
translation = translation.drop_duplicates().reset_index(drop=True)

geolocation: 1,000,163 -> 720,154 (280,009 dup coords)


In [74]:
len(reviews)

99224